In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display
from scipy import stats
from scipy.special import expit

In [ ]:
TASK = Path.cwd().parent.parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CONFIG = json.loads((ROOT / 'config.json').read_text())
EPS, Y, SEED = CONFIG['eps'], CONFIG['last_year'], CONFIG['seed']

In [ ]:
N = CONFIG['n_sim']
rng = np.random.default_rng(SEED)

In [ ]:
TAG = 'hurdle_ar1_s_p'
GENERATED_FROM = 1

In [ ]:
params = pd.read_csv(INPUT / TAG / 'parameters.csv', dtype={'stage':str})
panel = pd.read_csv(INPUT / 'data/panel.csv', index_col='dblp_id')
DEST = OUTPUT / TAG
DEST.mkdir(exist_ok=True)
sim = np.zeros((Y+1, N))

In [ ]:
initial = pd.read_csv(INPUT / TAG / 'initialization.csv').iloc[0]

In [ ]:
active0 = rng.random(N) < initial.p_init_active
sim[0,active0] = rng.exponential(initial.positive_scale, active0.sum())
for row in params.itertuples():
    for t in range(row.start, row.end+1):
        active = sim[t]>0
        next_active = np.zeros(N, dtype=bool)
        next_active[~active] = rng.random((~active).sum()) < row.p_restart
        p_drop = expit(row.dropout_intercept + row.dropout_log_q_coefficient*np.log(sim[t,active]+EPS))
        next_active[active] = rng.random(active.sum()) >= p_drop
        restart, continued = ~active & next_active, active & next_active
        sim[t+1,restart] = rng.exponential(initial.restart_scale, restart.sum())
        z = row.intercept + row.beta*np.log(sim[t,continued]) + rng.normal(0,row.sigma,continued.sum())
        sim[t+1,continued] = np.exp(z)

In [ ]:
np.save(DEST / 'trajectories.npy', sim, allow_pickle=False)
pd.DataFrame([{'model':TAG, 'seed':SEED, 'n_sim':N, 'n_years':Y+1, 'generated_from_age':GENERATED_FROM,
               'orientation':'age x scholar', 'minimum':sim.min(), 'maximum':sim.max(),
               'finite':np.isfinite(sim).all()}]).to_csv(DEST / 'metadata.csv', index=False)
display(pd.read_csv(DEST / 'metadata.csv'))